# Amazon Bedrock AgentCore를 활용한 Multi-Agent 관찰성

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime과 CloudWatch GenAI Observability를 사용하여 완전한 관찰성을 갖춘 multi-agent system을 구축하는 방법을 살펴봅니다.

다음 두 가지 아키텍처를 살펴봅니다.
1. **Part 1: Single Runtime** - 모든 에이전트가 하나의 Runtime에서 실행됨(더 간단하며 trace 통합)
2. **Part 2: Multi-Runtime** - 서로 다른 framework의 에이전트가 각각 별도의 Runtime에서 실행됨(trace 연결)

### 튜토리얼 세부 정보

| 정보 | 세부 정보 |
|:------------|:--------|
| 튜토리얼 유형 | 대화형 |
| 에이전트 유형 | Multi-Agent(Supervisor pattern) |
| Agentic Frameworks | Strands Agents & LangGraph |
| LLM 모델 | Anthropic Claude Haiku 4.5 |
| 주요 기능 | Multi-agent 조정, OTEL 관찰성 |
| 사용 SDK | Amazon BedrockAgentCore Python SDK, boto3, Strands Agents, LangGraph |

### 학습 내용

- Strands와 LangGraph로 multi-agent system을 구축하는 방법
- `Strands Agents`와 `LangGraph`에서 관찰성이 작동하는 방식
- session 전파에 OpenTelemetry baggage를 사용하는 방법
- 여러 Runtime의 trace를 연결하는 방법
- GenAI Observability dashboard에서 trace를 확인하고 분석하는 방법

---
**두 가지 아키텍처:**
1. **Part 1: Single Runtime** - 모든 에이전트가 하나의 Runtime에서 실행됨(trace 통합)
2. **Part 2: 여러 framework를 사용하는 Multi-Runtime** - 분산 에이전트(session 연결을 통해 trace 연결)

### 비교

| 항목 | Single Runtime | Multi-Runtime  |
|--------|----------------|---------------------|
| **아키텍처** | Runtime 1개, 모든 에이전트 | 별도 Runtime 3개 |
| **Frameworks** | 모두 Strands | Strands Agents + LangGraph |
| **통신** | 직접 함수 호출 | `runtimeSessionId`를 사용한 Runtime 호출 |
| **Session 전파** | Runtime 내부에서 자동 처리 | OpenTelemetry baggage |
| **IAM** | 단일 role | 에이전트별 별도 role |
| **Trace View** | 하나로 통합된 trace tree | session ID로 연결된 trace |
| **복잡도** | 더 간단함 | 더 복잡함 |
| **사용 사례** | 단일 팀, 긴밀한 통합 | 서로 다른 팀, 서로 다른 framework |

> **참고:** 이 튜토리얼은 학습 목적으로 **multi-agent architecture pattern**과 **관찰성 설정**을 보여 줍니다.

---

## 아키텍처 개요

```
                                사용자 요청
                                     │
                 ┌───────────────────┴───────────────────┐
                 │                                       │
                 ▼                                       ▼
┌────────────────────────────────────┐  ┌────────────────────────────────────┐
│     PART 1: SINGLE RUNTIME         │  │     PART 2: MULTI-RUNTIME          │
│     (간단한 통합 구성)             │  │     (분산 및 확장 가능 구성)        │
└────────────────────────────────────┘  └────────────────────────────────────┘

        SINGLE RUNTIME                          MULTI-RUNTIME 
┌──────────────────────────────┐       ┌──────────────────────────────┐
│   SINGLE AGENTCORE RUNTIME   │       │   ORCHESTRATOR (Strands)     │
│                              │       │   AgentCore Runtime #1       │
│  ┌────────────────────────┐  │       │   [OTel Baggage: session.id] │
│  │  ORCHESTRATOR (Strands)│  │       └──────────────┬───────────────┘
│  │         │              │  │                      │
│  │    ┌────┴────┐         │  │                      |
│  │    ▼         ▼         │  │              ┌───────┴───────┐
│  │ TRAVEL    WEATHER      │  │              │               │
│  │(Strands) (Strands)     │  │              ▼               ▼
│  │web_search get_weather  │  │       ┌─────────────┐ ┌─────────────┐
│  └────────────────────────┘  │       │TRAVEL AGENT │ │WEATHER AGENT│
│                              │       │ (Strands)   │ │ (LangGraph) │
│  → 하나로 통합된 trace       │       │ Runtime #2  │ │ Runtime #3  │
└──────────────────────────────┘       │ web_search  │ │ get_weather │
                                       └─────────────┘ └─────────────┘

          관찰성                                  관찰성
┌──────────────────────────────┐       ┌──────────────────────────────┐
│  하나로 통합된 Trace Tree    │       │  Session으로 연결된 Trace    │
│                              │       │                              │
│  Orchestrator                │       │                              │
│       │                      │       │                              │
│       ├── Travel Agent       │       │  Orchestrator Trace          │
│       │    └── web_search    │       │  Travel Agent Trace          │
│       └── Weather Agent      │       │  Weather Agent Trace         │
│            └── get_weather   │       │                              │
└──────────────────────────────┘       └──────────────────────────────┘
                 │                                       │
                 └───────────────────┬───────────────────┘
                                     ▼
                    ┌────────────────────────────────────┐
                    │  CloudWatch GenAI Observability    │
                    └────────────────────────────────────┘
```

## 사전 요구 사항

1. 필요한 최소 권한으로 AWS CLI 구성(`aws configure`)
2. Amazon Bedrock 모델 `global.anthropic.claude-haiku-4-5-20251001-v1:0` 사용 권한
3. CloudWatch [Transaction Search 활성화](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html)

## 설정

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import os
import boto3
import time
from pathlib import Path
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

# 변경되지 않는 절대 경로로 BASE_DIR 설정
BASE_DIR = Path(os.getcwd()).resolve()
print(f"Base directory: {BASE_DIR}")

# utility 가져오기
import sys

sys.path.insert(0, str(BASE_DIR))
from utils import (
    update_orchestrator_permissions,
    cleanup_runtime,
    cleanup_ssm_parameters,
)

region = Session().region_name
print(f"Using region: {region}")

배포가 완료될 때까지 Runtime 상태를 polling하는 helper입니다.

In [ ]:
def wait_for_ready(runtime):
    """런타임이 준비될 때까지 기다립니다."""
    status = runtime.status().endpoint["status"]
    print(status)
    while status not in ["READY", "CREATE_FAILED", "UPDATE_FAILED"]:
        time.sleep(10)
        status = runtime.status().endpoint["status"]
        print(f"Status: {status}")
    return status

---
# Part 1: Single Runtime Multi-Agent

모든 에이전트(Orchestrator, Travel, Weather)가 직접 함수 호출을 사용하여 **하나의 Runtime**에서 실행됩니다.

이 섹션에서는 모든 에이전트(Orchestrator, Travel, Weather)를 **단일 AgentCore Runtime**에 배포합니다.

**아키텍처:**
```
┌─────────────────────────────────────┐
│     Single AgentCore Runtime        │
│  ┌─────────────────────────────┐    │
│  │    ORCHESTRATOR (Strands)   │    │
│  │         │                   │    │
│  │    ┌────┴────┐              │    │
│  │    ▼         ▼              │    │
│  │ TRAVEL    WEATHER           │    │
│  │ (Strands) (Strands)         │    │
│  │ web_search get_weather      │    │
│  └─────────────────────────────┘    │
│                                     │
│                                     │
│  → 하나로 통합된 trace tree         │
└─────────────────────────────────────┘
```

- **이점**: 간단한 설정, CloudWatch에서 하나로 통합된 trace tree

단일 Runtime을 배포합니다. starter toolkit은 다음 작업을 처리합니다.
- Docker image를 build하여 ECR에 push
- 필요한 권한을 갖춘 IAM execution role 생성
- AgentCore Runtime에 배포

In [ ]:
# single_runtime 디렉터리로 이동
os.chdir(BASE_DIR / "single_runtime")

single_runtime = Runtime()
single_runtime.configure(
    entrypoint="multi_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="single_runtime_demo",
)

single_launch = single_runtime.launch()
print(f"Agent ARN: {single_launch.agent_arn}")

# base 디렉터리로 돌아가기
os.chdir(BASE_DIR)

In [ ]:
wait_for_ready(single_runtime)

### Single Runtime 테스트

In [ ]:
# 호출 1
print("=== Invocation 1 ===")
response = single_runtime.invoke({"prompt": "What's the weather in Paris and what should I visit?"})
print(response)

In [ ]:
# 호출 2
print("=== Invocation 2 ===")
response = single_runtime.invoke({"prompt": "What's the weather in Tokyo and recommend some local food?"})
print(response)

### CloudWatch GenAI Observability Dashboard에서 trace 확인

<div style="text-align:left">
    <img src="images/single_runtime.png" width="50%"/>
</div>


---
# Part 2: Multi-Runtime

서로 분리된 세 개의 Runtime이 `invoke_agent_runtime()`을 통해 통신합니다.

```
┌───────────────────────────────────────────────────────────┐
│                  ORCHESTRATOR (Strands)                   │
│                  AgentCore Runtime #1                     │
└─────────────────────────┬─────────────────────────────────┘
                          │ invoke_agent_runtime()
          ┌───────────────┴───────────────┐
          ▼                               ▼
┌───────────────────────┐     ┌───────────────────────┐
│   TRAVEL (Strands)    │     │  WEATHER (LangGraph)  │
│   Runtime #2          │     │  Runtime #3           │
│   web_search          │     │                       │
└───────────────────────┘     └───────────────────────┘
```

### 2.1 AgentCore Runtime에 Travel Agent(Strands) 배포

In [ ]:
# travel_agent 디렉터리로 이동
os.chdir(BASE_DIR / "travel_agent")

안전하게 저장하고 orchestrator가 검색할 수 있도록 agent ARN을 SSM에 저장합니다.

In [ ]:
travel_runtime = Runtime()
travel_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="travel_subagent_strands",
)

travel_launch = travel_runtime.launch()
travel_arn = travel_launch.agent_arn
print(f"Travel Agent ARN: {travel_arn}")

In [ ]:
ssm = boto3.client("ssm")
ssm.put_parameter(Name="/agents/travel_agent_arn", Value=travel_arn, Type="String", Overwrite=True)
ssm.put_parameter(
    Name="/agents/travel_agent_provider",
    Value="travel_agent_strands",
    Type="String",
    Overwrite=True,
)
print("Travel Agent metadata saved to SSM")

In [ ]:
wait_for_ready(travel_runtime)

print("\n" + "=" * 80)
print("📊 ENABLE OBSERVABILITY FOR TRAVEL AGENT RUNTIME")
print("=" * 80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console and ensure you have Transaction Search enabled in your account:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'travel_subagent_strands'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/travel_subagent_strands
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save
""")

# base 디렉터리로 돌아가기
os.chdir(BASE_DIR)

<div style="text-align:left">
    <img src="images/vended_logs_trace_runtime.png" width="50%"/>
</div>

### 2.2 AgentCore Runtime에 Weather Agent(LangGraph) 배포


In [ ]:
# weather_agent 디렉터리로 이동
os.chdir(BASE_DIR / "weather_agent")

weather_runtime = Runtime()
weather_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="weather_subagent_lang",
)

weather_launch = weather_runtime.launch()
weather_arn = weather_launch.agent_arn
print(f"Weather Agent ARN: {weather_arn}")

In [ ]:
ssm.put_parameter(Name="/agents/weather_agent_arn", Value=weather_arn, Type="String", Overwrite=True)
ssm.put_parameter(
    Name="/agents/weather_agent_provider",
    Value="weather_agent_lang",
    Type="String",
    Overwrite=True,
)
print("Weather Agent metadata saved to SSM")

In [ ]:
wait_for_ready(weather_runtime)

print("\n" + "=" * 80)
print("📊 ENABLE OBSERVABILITY FOR WEATHER AGENT RUNTIME")
print("=" * 80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'weather_agent_lang'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/weather_agent_lang
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save

""")

# base 디렉터리로 돌아가기
os.chdir(BASE_DIR)

### 2.3 Orchestrator Agent 배포

session ID를 전파하면서 `invoke_agent_runtime()`을 통해 query를 sub-agent로 routing합니다.

In [ ]:
# orchestrator_agent 디렉터리로 이동
os.chdir(BASE_DIR / "orchestrator_agent")

orchestrator_runtime = Runtime()
orchestrator_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="orchestrator_strands",
)

orchestrator_launch = orchestrator_runtime.launch()
print(f"Orchestrator ARN: {orchestrator_launch.agent_arn}")

In [ ]:
# 먼저 다음 값을 확인
print(f"Travel ARN: {travel_arn}")
print(f"Weather ARN: {weather_arn}")
print(f"Orchestrator ID: {orchestrator_launch.agent_id}")

orchestrator가 sub-agent를 호출하는 데 필요한 IAM 권한을 추가합니다.

**중요**: IAM 권한을 업데이트한 후 변경 사항이 AWS 서비스 전체에 전파되도록 60초 동안 기다립니다. 이렇게 하면 orchestrator가 SSM parameter에 액세스하거나 sub-agent를 호출할 때 발생할 수 있는 "AccessDeniedException" 오류를 방지할 수 있습니다.

In [ ]:
# base 디렉터리에서 utils 가져오기
import sys

sys.path.insert(0, str(BASE_DIR))

update_orchestrator_permissions(
    sub_agent_arns=[travel_arn, weather_arn],
    orchestrator_agent_id=orchestrator_launch.agent_id,
    region=region,
)

# IAM 권한 전파 대기
print("⏳ Waiting 60 seconds for IAM permissions to propagate...")
print("   This ensures the orchestrator can access SSM parameters and invoke sub-agents.")
time.sleep(60)
print("✅ IAM propagation wait complete")

In [ ]:
wait_for_ready(orchestrator_runtime)

print("\n" + "=" * 80)
print("📊 ENABLE OBSERVABILITY FOR ORCHESTRATOR RUNTIME")
print("=" * 80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'orchestrator_strands'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/orchestrator_strands
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save

""")

# base 디렉터리로 돌아가기
os.chdir(BASE_DIR)

### Multi-Runtime, Multi Framework 설정 테스트

In [ ]:
# 관찰성 설정을 활성화할 수 있도록 일시 중지
print("⏸️  IMPORTANT: Before testing the multi-runtime system:")
print("1. Enable vended logs and tracing for all three runtimes in the AWS Console")
print("2. Follow the instructions printed above for each runtime")
print("3. Wait 1-2 minutes for settings to propagate")
print("\n✅ Once you've enabled observability for all runtimes, proceed to test the system")

### 📊 AgentCore Observability 구성 요약

세 Runtime 모두에 vended logs와 tracing을 활성화하고 Agent Observability를 구성하면 에이전트 동작과 함께 시스템 상태를 확인할 수 있습니다.

| Runtime | Log Group | Tracing |
|---------|-----------|---------|
| Travel Agent | `/aws/vendedlogs/bedrock-agentcore/travel_subagent_strands` | Enabled |
| Weather Agent | `/aws/vendedlogs/bedrock-agentcore/weather_subagent_lang` | Enabled |
| Orchestrator | `/aws/vendedlogs/bedrock-agentcore/orchestrator_strands` | Enabled |

**주요 이점:**
- **Vended Logs**: 각 Runtime의 application log가 CloudWatch로 자동 전송됨
- **Distributed Tracing**: 여러 Runtime의 trace가 서로 연결됨
- **GenAI Observability**: multi-agent 상호작용을 전체적으로 파악 가능

In [ ]:
# 호출 1
print("=== Invocation 1 ===")
response = orchestrator_runtime.invoke({"prompt": "What's the weather in New York and what museums should I visit?"})
print(response)

print("\n" + "=" * 50 + "\n")

# 호출 2
print("=== Invocation 2 ===")
response = orchestrator_runtime.invoke(
    {"prompt": "What's the weather in Seattle and what are the best parks to visit?"}
)
print(response)

### Trace 확인

multi-agent system의 trace와 관찰성 데이터를 확인하는 방법은 다음과 같습니다.

## Amazon CloudWatch의 AgentCore Observability

모든 Runtime에서 vended logs와 tracing을 활성화한 후 다음 단계에 따라 관찰성 데이터를 확인합니다.


- 세 Runtime 모두에서 vended logs와 tracing이 활성화되어 있어야 합니다.
- 호출 후 trace가 나타날 때까지 2~3분 기다립니다.

### GenAI Observability Dashboard에서 Multi-Agent System 확인

#### 1. **Bedrock AgentCore 개요**
**AWS Console** → **CloudWatch** → **GenAI Observability**로 이동합니다.

모든 에이전트가 표시됩니다.
- `orchestrator_strands`
- `travel_subagent_strands`
- `weather_agent_lang`

최근 테스트 호출에 집중할 수 있도록 기간별로 데이터를 필터링합니다.

#### 2. **Runtime Metric(모든 에이전트)**
기본 dashboard에서 모든 에이전트의 Runtime metric을 확인합니다.

<div style="text-align:left">
    <img src="images/runtime_metrics.png" width="50%"/>
</div>


#### 3. **에이전트별 View**
특정 에이전트(예: `orchestrator_strands`)를 클릭하면 다음 정보를 확인할 수 있습니다.
- 에이전트별 Runtime metric
- request/response pattern
- 성능 변화
- 사용자 지정 기간 필터링

<div style="text-align:left">
    <img src="images/per_agent.png" width="50%"/>
</div>

#### 4. **Sessions View**
**Sessions View** 탭으로 이동하면 다음 정보를 확인할 수 있습니다.
- 각 에이전트와 연결된 모든 session
- 여러 Runtime의 request를 연결하는 session ID
- orchestrator에서 sub-agent로 이어지는 request flow

#### 5. **Trace View**
**Trace View** 탭에서 다음 항목을 살펴봅니다.
- timeline에 표시된 각 Runtime의 상세 trace 및 span 정보
- 전체 trajectory view
  ```
  Orchestrator (receives request)
  ├── Invoke Travel Agent (with runtimeSessionId)
  │   └── Web Search Tool calls
  ├── Invoke Weather Agent (with runtimeSessionId)
  │   └── Weather Tool calls
  └── Response aggregation
  ```
- 다음 정보를 보여 주는 span attribute:
  - 모델 호출
  - Tool 실행
  - latency 세부 내역


<div style="text-align:left">
    <img src="images/full_trace.png" width="80%"/>
</div>


#### 6. **Runtime 간 연결된 Trace**
전체 multi-agent 상호작용을 확인하려면 다음 단계를 수행합니다.
1. orchestrator의 trace를 찾습니다.
2. 세 Runtime의 연결된 trace에서 전체 분산 실행을 확인합니다. sub-agent의 trace도 확인할 수 있습니다.

### Multi-Agent System의 주요 관찰성 기능

| 기능 | 설명 | 사용 사례 |
|---------|-------------|----------|
| **Vended Logs** | application log를 CloudWatch로 자동 전송 | 에이전트 logic 및 오류 디버깅 |
| **Distributed Tracing** | trace 연결 | 여러 에이전트의 request 추적 |
| **Session Correlation** | 단일 사용자 request의 모든 span 연결 | 전체 request flow 파악 |
| **Tool Execution Spans** | 개별 tool 호출(`web_search`, `get_weather`) 확인 | 성능 투명성 확보 |
| **Model Invocation Metrics** | 모델 호출별 token 사용량 및 latency | 비용 및 성능 모니터링 |

### 에이전트와 AgentCore Runtime의 Vended Logs 및 Tracing

GenAI Observability dashboard의 여러 기능을 살펴보며 상세 trace 정보, 성능 metric 및 시스템 동작을 확인하세요.

---
# 정리

ECR, log 등으로 인한 추가 비용이 발생하지 않도록 관련 리소스를 모두 정리하세요.

In [ ]:
# 모든 리소스 정리
cleanup_runtime(single_launch, "single_runtime_demo", region)
cleanup_runtime(travel_launch, "travel_agent_strands", region)
cleanup_runtime(weather_launch, "weather_agent_lang", region)
cleanup_runtime(orchestrator_launch, "orchestrator_strands", region)

cleanup_ssm_parameters(
    [
        "/agents/travel_agent_arn",
        "/agents/travel_agent_provider",
        "/agents/weather_agent_arn",
        "/agents/weather_agent_provider",
    ]
)

print("Cleanup complete!")